# Demand Forecasting – EDA & Modellierung

**Datensatz:** `demand_forecasting.csv`  
**Zeitraum:** Januar 2022 – Januar 2024  
**Zielvariable:** `demand`

---
## Inhaltsverzeichnis
1. [Setup & Daten laden](#1-setup)
2. [Datenbereinigung & Preprocessing](#2-preprocessing)
3. [Explorative Datenanalyse (EDA)](#3-eda)
   - 3.1 Übersicht & Datenqualität
   - 3.2 Zielvariable `demand`
   - 3.3 Zeitliche Analyse
   - 3.4 Kategorische Features
   - 3.5 Numerische Features & Korrelationen
   - 3.6 Lost-Sales-Analyse (Stockouts)
4. [Feature Engineering](#4-features)
5. [Modellierung mit XGBoost](#5-modell)
6. [Modell-Evaluation](#6-evaluation)
7. [Feature Importance](#7-importance)

---
## 1. Setup & Daten laden <a id='1-setup'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor, plot_importance

# Reproduzierbarkeit
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot-Stil
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("✅ Setup abgeschlossen")

In [ ]:
df = pd.read_csv("data/demand_forecasting.csv")
print(f"Shape: {df.shape}")
df.head()

---
## 2. Datenbereinigung & Preprocessing <a id='2-preprocessing'></a>

In [ ]:
# Spaltennamen normalisieren
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)

# String-Spalten bereinigen
str_cols = ["store_id", "product_id", "category", "region", "seasonality", "weather_condition"]
for col in str_cols:
    df[col] = df[col].str.lower().str.strip()

# Datum konvertieren und Zeitfeatures extrahieren
df["date"] = pd.to_datetime(df["date"])
df["year"]    = df["date"].dt.year
df["month"]   = df["date"].dt.month
df["day"]     = df["date"].dt.day
df["weekday"] = df["date"].dt.day_name()
df["is_weekend"] = df["date"].dt.dayofweek.isin([5, 6]).astype(int)

print(f"Shape nach Preprocessing: {df.shape}")
print(f"Zeitraum: {df['date'].min().date()} bis {df['date'].max().date()}")
df.dtypes

---
## 3. Explorative Datenanalyse (EDA) <a id='3-eda'></a>
### 3.1 Übersicht & Datenqualität

In [ ]:
print("=== Shape ===")
print(f"{df.shape[0]:,} Zeilen, {df.shape[1]} Spalten\n")

print("=== Fehlende Werte ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "Keine fehlenden Werte ✅")

print("\n=== Duplikate ===")
dupes = df.duplicated().sum()
print(f"{dupes} Duplikate" + (" ✅" if dupes == 0 else " ⚠️"))

In [ ]:
# Numerische Statistiken
num_cols = ["inventory_level", "units_sold", "units_ordered",
            "price", "discount", "competitor_pricing", "demand"]
df[num_cols].describe().round(2).T

In [ ]:
# Kategorische Übersicht
# FIX: include="str" statt "object" (Pandas 4 Kompatibilität)
df.describe(include="str").T

In [ ]:
# Unique-Werte der kategorischen Spalten
for col in ["category", "region", "weather_condition", "seasonality"]:
    print(f"{col}: {sorted(df[col].unique())}")

### 3.2 Zielvariable `demand`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramm
axes[0].hist(df["demand"], bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(df["demand"].mean(), color="red", linestyle="--", label=f'Mean: {df["demand"].mean():.1f}')
axes[0].axvline(df["demand"].median(), color="orange", linestyle="--", label=f'Median: {df["demand"].median():.1f}')
axes[0].set_title("Verteilung der Zielvariable 'demand'")
axes[0].set_xlabel("Demand")
axes[0].set_ylabel("Häufigkeit")
axes[0].legend()

# Boxplot
axes[1].boxplot(df["demand"], vert=True, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.7))
axes[1].set_title("Boxplot 'demand'")
axes[1].set_ylabel("Demand")
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

# Kennzahlen
print(f"Min: {df['demand'].min()}")
print(f"Max: {df['demand'].max()}")
print(f"Mean: {df['demand'].mean():.2f}")
print(f"Median: {df['demand'].median():.2f}")
print(f"Std: {df['demand'].std():.2f}")
print(f"Skewness: {df['demand'].skew():.3f}  → leicht rechtschief")

In [ ]:
# Demand vs. Units Sold: Lost Sales Indikator
df["lost_sales"] = df["demand"] - df["units_sold"]
df["has_lost_sales"] = (df["lost_sales"] > 0).astype(int)

pct_lost = df["has_lost_sales"].mean() * 100
avg_lost = df.loc[df["has_lost_sales"] == 1, "lost_sales"].mean()

print(f"Anteil Zeilen mit Lost Sales (demand > units_sold): {pct_lost:.1f}%")
print(f"Ø Lost Sales pro betroffener Zeile: {avg_lost:.1f} Einheiten")
print(f"\nGesamt demand:        {df['demand'].sum():>12,}")
print(f"Gesamt units_sold:    {df['units_sold'].sum():>12,}")
print(f"Gesamt lost sales:    {df['lost_sales'].clip(lower=0).sum():>12,}")

### 3.3 Zeitliche Analyse

In [ ]:
# Monatliche Nachfrage pro Jahr (Line-Chart)
df_monthly = df.groupby(["year", "month"])["demand"].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#2196F3", "#FF5722", "#4CAF50"]

for i, year in enumerate(sorted(df_monthly["year"].unique())):
    subset = df_monthly[df_monthly["year"] == year]
    ax.plot(subset["month"], subset["demand"],
            marker="o", label=str(year), color=colors[i], linewidth=2.5)

ax.set_title("Monatliche Gesamtnachfrage nach Jahr")
ax.set_xlabel("Monat")
ax.set_ylabel("Demand (Summe)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Feb","Mär","Apr","Mai","Jun",
                    "Jul","Aug","Sep","Okt","Nov","Dez"])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend(title="Jahr")
plt.tight_layout()
plt.show()

In [ ]:
# Nachfrage nach Saison
season_order = ["spring", "summer", "autumn", "winter"]
season_demand = df.groupby("seasonality")["demand"].agg(["mean", "sum"]).reindex(season_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(season_demand.index, season_demand["mean"], color=["#66BB6A","#FFA726","#EF5350","#42A5F5"])
axes[0].set_title("Ø Demand nach Saison")
axes[0].set_ylabel("Ø Demand")

axes[1].bar(season_demand.index, season_demand["sum"] / 1e6, color=["#66BB6A","#FFA726","#EF5350","#42A5F5"])
axes[1].set_title("Gesamt-Demand nach Saison")
axes[1].set_ylabel("Demand (Mio.)")

plt.tight_layout()
plt.show()

In [ ]:
# Nachfrage nach Wochentag
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_demand = df.groupby("weekday")["demand"].mean().reindex(weekday_order)

colors_wd = ["#90A4AE"] * 5 + ["#42A5F5", "#42A5F5"]  # Wochenende hervorheben
plt.figure(figsize=(12, 4))
plt.bar(weekday_demand.index, weekday_demand.values, color=colors_wd, edgecolor="white")
plt.title("Ø Demand nach Wochentag (blau = Wochenende)")
plt.ylabel("Ø Demand")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 3.4 Kategorische Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

cat_features = {
    "category": axes[0, 0],
    "region": axes[0, 1],
    "weather_condition": axes[1, 0],
    "seasonality": axes[1, 1]
}

for feat, ax in cat_features.items():
    order = df.groupby(feat)["demand"].mean().sort_values(ascending=False).index
    sns.boxplot(data=df, x=feat, y="demand", order=order, ax=ax, palette="muted")
    ax.set_title(f"Demand nach '{feat}'")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Promotion & Epidemic Effekt
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, label in zip(axes, ["promotion", "epidemic"], ["Promotion", "Epidemic"]):
    means = df.groupby(col)["demand"].mean()
    ax.bar(["Nein (0)", "Ja (1)"], means.values, color=["#90CAF9", "#EF5350"])
    ax.set_title(f"Ø Demand: {label}")
    ax.set_ylabel("Ø Demand")
    for i, v in enumerate(means.values):
        ax.text(i, v + 0.5, f"{v:.1f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Pivot-Tabelle: Ø Demand nach Jahr/Monat und Kategorie
df.pivot_table(
    index=["year", "month"],
    columns="category",
    values="demand",
    aggfunc="mean"
).round(1)

### 3.5 Numerische Features & Korrelationen

In [ ]:
# Korrelationsmatrix
corr_cols = ["inventory_level", "units_sold", "units_ordered",
             "price", "discount", "competitor_pricing",
             "promotion", "epidemic", "demand"]

corr = df[corr_cols].corr()

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True
)
plt.title("Korrelationsmatrix – numerische Features")
plt.tight_layout()
plt.show()

# Korrelationen mit Zielvariable (sortiert)
print("\nKorrelation mit 'demand':")
print(corr["demand"].drop("demand").sort_values(ascending=False).round(3))

In [ ]:
# Scatter: Preis vs. Demand (nach Kategorie eingefärbt)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Eigener Preis vs. Demand
for cat in df["category"].unique():
    mask = df["category"] == cat
    axes[0].scatter(df.loc[mask, "price"], df.loc[mask, "demand"],
                    alpha=0.15, s=5, label=cat)
axes[0].set_title("Eigener Preis vs. Demand")
axes[0].set_xlabel("Price")
axes[0].set_ylabel("Demand")
axes[0].legend(markerscale=4, fontsize=8)

# Wettbewerberpreis vs. Demand
axes[1].scatter(df["competitor_pricing"], df["demand"],
                alpha=0.15, s=5, color="coral")
axes[1].set_title("Wettbewerberpreis vs. Demand")
axes[1].set_xlabel("Competitor Pricing")
axes[1].set_ylabel("Demand")

plt.tight_layout()
plt.show()

In [ ]:
# Rabatt-Analyse
df["discount_bin"] = pd.cut(df["discount"],
                             bins=[-1, 0, 5, 10, 15, 25],
                             labels=["0%", "1–5%", "6–10%", "11–15%", "16–25%"])

discount_demand = df.groupby("discount_bin", observed=True)["demand"].mean()

plt.figure(figsize=(9, 4))
plt.bar(discount_demand.index.astype(str), discount_demand.values,
        color="#5C6BC0", edgecolor="white")
plt.title("Ø Demand nach Rabatt-Kategorie")
plt.xlabel("Rabatt")
plt.ylabel("Ø Demand")
for i, v in enumerate(discount_demand.values):
    plt.text(i, v + 0.3, f"{v:.1f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

### 3.6 Lost-Sales-Analyse (Stockouts)

In [ ]:
# Artikel mit Lagerbestand = 0 (Stockouts)
stockouts = df[df["inventory_level"] == 0]
print(f"Stockout-Zeilen: {len(stockouts):,} ({len(stockouts)/len(df)*100:.1f}% aller Zeilen)")
print("\nStockouts nach Kategorie:")
print(stockouts.groupby("category").size().sort_values(ascending=False))

# Stockout-Rate pro Kategorie
stockout_rate = df.groupby("category").apply(
    lambda x: (x["inventory_level"] == 0).mean() * 100
).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
plt.bar(stockout_rate.index, stockout_rate.values, color="#EF5350", edgecolor="white")
plt.title("Stockout-Rate nach Kategorie (Inventory = 0)")
plt.ylabel("Stockout-Rate (%)")
plt.xlabel("Kategorie")
plt.tight_layout()
plt.show()

In [ ]:
# Lost Sales nach Kategorie
lost_by_cat = df.groupby("category")["lost_sales"].agg(
    total_lost=lambda x: x.clip(lower=0).sum(),
    pct_rows_affected=lambda x: (x > 0).mean() * 100
).sort_values("total_lost", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(lost_by_cat.index, lost_by_cat["total_lost"],
             color="#FF7043", edgecolor="white")
axes[0].set_title("Gesamt Lost Sales nach Kategorie")
axes[0].set_xlabel("Lost Sales (Einheiten)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[1].barh(lost_by_cat.index, lost_by_cat["pct_rows_affected"],
             color="#FFA726", edgecolor="white")
axes[1].set_title("% Zeilen mit Lost Sales nach Kategorie")
axes[1].set_xlabel("Anteil (%)")

plt.tight_layout()
plt.show()

---
## 4. Feature Engineering <a id='4-features'></a>

In [ ]:
# Daten neu laden (saubere Basis für Modellierung)
df_model = pd.read_csv("data/demand_forecasting.csv")
df_model.columns = df_model.columns.str.lower().str.replace(" ", "_")

# String bereinigen
for col in ["store_id", "product_id", "category", "region",
            "seasonality", "weather_condition"]:
    df_model[col] = df_model[col].str.lower().str.strip()

# Datum
df_model["date"] = pd.to_datetime(df_model["date"])

# === Neue Features ===

# 1. Preisposition gegenüber Wettbewerber
df_model["price_diff"] = df_model["price"] - df_model["competitor_pricing"]
df_model["price_ratio"] = df_model["price"] / (df_model["competitor_pricing"] + 1e-6)

# 2. Effektiver Preis nach Rabatt
df_model["effective_price"] = df_model["price"] * (1 - df_model["discount"] / 100)

# 3. Lagerbestandsrisiko
df_model["stockout_flag"] = (df_model["inventory_level"] == 0).astype(int)
df_model["low_stock_flag"] = (df_model["inventory_level"] < 50).astype(int)

# 4. Zeitfeatures
df_model["month"] = df_model["date"].dt.month
df_model["year"]  = df_model["date"].dt.year
df_model["is_weekend"] = df_model["date"].dt.dayofweek.isin([5, 6]).astype(int)

print("Neue Features erstellt:")
new_features = ["price_diff", "price_ratio", "effective_price",
                "stockout_flag", "low_stock_flag", "month", "year", "is_weekend"]
print(df_model[new_features].describe().round(3))

---
## 5. Modellierung mit XGBoost <a id='5-modell'></a>

> ⚠️ **Hinweis zu Data Leakage:** `units_sold` ist eine Realisierung von `demand` und darf **nicht** als Feature verwendet werden. Im produktiven Einsatz wäre `units_sold` zur Vorhersagezeit unbekannt.

In [ ]:
# Features und Zielvariable definieren
# WICHTIG: units_sold NICHT als Feature (Data Leakage!)
features = [
    "price",
    "discount",
    "effective_price",
    "inventory_level",
    "stockout_flag",
    "low_stock_flag",
    "promotion",
    "epidemic",
    "competitor_pricing",
    "price_diff",
    "price_ratio",
    "category",
    "region",
    "weather_condition",
    "seasonality",
    "month",
    "year",
    "is_weekend",
]

target = "demand"

X = df_model[features].copy()
y = df_model[target].copy()

print(f"Feature-Matrix: {X.shape}")
print(f"Zielvariable: {y.shape}")

In [ ]:
# Label Encoding der kategorischen Spalten
# FIX: include="str" für Pandas 4 Kompatibilität
categorical_cols = X.select_dtypes(include="str").columns.tolist()
print(f"Kategorische Spalten: {categorical_cols}")

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

X.head()

In [ ]:
# Zeitbasierter Train/Test-Split
# FIX: Kein zufälliger Split bei Zeitreihendaten (vermeidet Leakage durch Zeitkomponente)
split_date = "2024-01-01"

train_mask = df_model["date"] < split_date
test_mask  = df_model["date"] >= split_date

X_train, X_test = X[train_mask].reset_index(drop=True), X[test_mask].reset_index(drop=True)
y_train, y_test = y[train_mask].reset_index(drop=True), y[test_mask].reset_index(drop=True)

print(f"Train: {X_train.shape[0]:,} Zeilen ({train_mask.sum()/len(df_model)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]:,} Zeilen ({test_mask.sum()/len(df_model)*100:.1f}%)")
print(f"\nTrain-Zeitraum: {df_model.loc[train_mask, 'date'].min().date()} – {df_model.loc[train_mask, 'date'].max().date()}")
print(f"Test-Zeitraum:  {df_model.loc[test_mask, 'date'].min().date()} – {df_model.loc[test_mask, 'date'].max().date()}")

In [ ]:
# XGBoost mit RandomizedSearchCV
# FIX: random_state für Reproduzierbarkeit
xgb = XGBRegressor(
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

param_dict = {
    "n_estimators":  [200, 300, 500],
    "max_depth":     [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample":     [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dict,
    n_iter=25,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

random_search.fit(X_train, y_train)

print("\n✅ Training abgeschlossen")
print(f"Beste Parameter: {random_search.best_params_}")
print(f"Bester CV-MAE: {-random_search.best_score_:.3f}")

---
## 6. Modell-Evaluation <a id='6-evaluation'></a>

In [ ]:
best_model = random_search.best_estimator_

y_pred_train = best_model.predict(X_train)
y_pred_test  = best_model.predict(X_test)

def eval_metrics(y_true, y_pred, label=""):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-6))) * 100
    print(f"{'='*40}")
    print(f"  {label}")
    print(f"{'='*40}")
    print(f"  MAE:  {mae:>10.3f}")
    print(f"  RMSE: {rmse:>10.3f}")
    print(f"  R²:   {r2:>10.4f}")
    print(f"  MAPE: {mape:>9.2f}%")
    return {"mae": mae, "rmse": rmse, "r2": r2, "mape": mape}

train_metrics = eval_metrics(y_train, y_pred_train, "TRAIN")
test_metrics  = eval_metrics(y_test,  y_pred_test,  "TEST")

In [ ]:
# Residual-Analyse
residuals = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Actual vs. Predicted
axes[0].scatter(y_test, y_pred_test, alpha=0.3, s=8, color="steelblue")
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, "r--", linewidth=1.5, label="Ideal")
axes[0].set_title("Actual vs. Predicted")
axes[0].set_xlabel("Actual Demand")
axes[0].set_ylabel("Predicted Demand")
axes[0].legend()

# 2. Residuals vs. Predicted
axes[1].scatter(y_pred_test, residuals, alpha=0.3, s=8, color="coral")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Residuals vs. Predicted")
axes[1].set_xlabel("Predicted Demand")
axes[1].set_ylabel("Residual")

# 3. Residual-Verteilung
axes[2].hist(residuals, bins=60, color="#5C6BC0", edgecolor="white")
axes[2].axvline(0, color="red", linestyle="--")
axes[2].set_title("Residual-Verteilung")
axes[2].set_xlabel("Residual")
axes[2].set_ylabel("Häufigkeit")

plt.tight_layout()
plt.show()

print(f"Residual-Statistiken:")
print(f"  Mean:   {residuals.mean():.3f}")
print(f"  Std:    {residuals.std():.3f}")
print(f"  Min:    {residuals.min():.3f}")
print(f"  Max:    {residuals.max():.3f}")

---
## 7. Feature Importance <a id='7-importance'></a>

In [ ]:
# Feature Importance (gain-basiert)
importance = best_model.get_booster().get_score(importance_type="gain")
imp_df = pd.DataFrame(
    {"feature": list(importance.keys()), "gain": list(importance.values())}
).sort_values("gain", ascending=True)

plt.figure(figsize=(10, 7))
colors = ["#42A5F5" if i >= len(imp_df) - 5 else "#90A4AE" for i in range(len(imp_df))]
plt.barh(imp_df["feature"], imp_df["gain"], color=colors, edgecolor="white")
plt.title("Feature Importance (Gain) – Top Features hervorgehoben")
plt.xlabel("Gain")
plt.tight_layout()
plt.show()

print("\nTop 10 Features nach Gain:")
print(imp_df.sort_values("gain", ascending=False).head(10).to_string(index=False))

In [ ]:
# Zusammenfassung
print("="*50)
print("  MODELL-ZUSAMMENFASSUNG")
print("="*50)
print(f"  Modell:     XGBoost Regressor")
print(f"  Features:   {len(features)}")
print(f"  Train-Rows: {X_train.shape[0]:,}")
print(f"  Test-Rows:  {X_test.shape[0]:,}")
print()
print(f"  Test-MAE:   {test_metrics['mae']:.3f}")
print(f"  Test-RMSE:  {test_metrics['rmse']:.3f}")
print(f"  Test-R²:    {test_metrics['r2']:.4f}")
print(f"  Test-MAPE:  {test_metrics['mape']:.2f}%")
print()
print(f"  Beste Params: {random_search.best_params_}")
print("="*50)